In [ ]:
# CELL 1 - mount Drive + install required libs
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

!pip install -q datasets transformers pandas scikit-learn


In [ ]:
# CELL 2 - preprocessing: load HF dataset, create train/val splits, merge train+test,
# convert output dict -> multiline risk report text (short natural fillers),
# and save train_all_951.jsonl, train.jsonl, val.jsonl, test.jsonl

import json
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive"
OUT_DIR = f"{DRIVE_BASE}/investlm_data_v3"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# ---------- 1) load original gretel dataset ----------
ds = load_dataset("gretelai/gretel-financial-risk-analysis-v1")
train = ds["train"]
test  = ds["test"]
print("Original sizes -> train:", len(train), "test:", len(test))

# convert to DataFrame for stratified split
train_df = pd.DataFrame(train)

# ---------- 2) create ~10% validation (stratified on risk_severity where possible) ----------
train_df["risk_severity"] = train_df["risk_severity"].fillna("UNKNOWN")
train_df_train, train_df_val = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["risk_severity"]
)
train_split = Dataset.from_pandas(train_df_train, preserve_index=False)
val_split   = Dataset.from_pandas(train_df_val,   preserve_index=False)
print("After split -> train_split:", len(train_split), "val_split:", len(val_split))

# ---------- 3) new instruction (exact text to be used in dataset) ----------
INSTRUCTION = (
    "You are a financial risk analysis model.\n"
    "Read the company text and generate a concise, structured risk report using EXACTLY the following fields and format, one field per line:\n\n"
    "risk_severity: <NONE | LOW | MEDIUM | HIGH>\n"
    "risk_categories: <semi-colon separated list, e.g., LIQUIDITY; DEBT; INTEREST_RATE>\n"
    "financial_impact: <short description of the potential or realized financial impact>\n"
    "key_metrics: <short description of important financial figures mentioned (amounts, debt levels, ratios, rates, etc.)>\n"
    "critical_dates: <important dates mentioned, or 'No specific critical dates disclosed.'>\n"
    "analysis: <2–4 sentence narrative summarizing the key risks and context>\n\n"
    "Rules:\n"
    "- ALWAYS include every field.\n"
    "- Output MUST be plain text, no JSON or brackets.\n"
    "- Do NOT add quotes, bullets, or extra commentary.\n"
    "- Write clearly and professionally."
)

# ---------- 4) helper to convert `output` dict -> multiline report (short natural fillers) ----------
def dict_to_risk_text(output_obj):
    # output_obj may be dict or JSON string
    if isinstance(output_obj, str):
        try:
            output_obj = json.loads(output_obj)
        except Exception:
            output_obj = {"analysis": output_obj}

    out = dict(output_obj) if isinstance(output_obj, dict) else {}

    # short natural fillers when missing
    risk_severity = (out.get("risk_severity") or "NONE")
    # normalize to uppercase / canonical (if provided)
    if isinstance(risk_severity, str):
        risk_severity = risk_severity.strip().upper()
        if risk_severity == "":
            risk_severity = "NONE"

    risk_categories = out.get("risk_categories") or []
    if isinstance(risk_categories, str):
        # sometimes categories are comma-separated; keep as list
        rc = [c.strip() for c in risk_categories.replace(",", ";").split(";") if c.strip()]
        risk_categories = rc

    financial_impact = out.get("financial_impact", None)
    if financial_impact is None or (isinstance(financial_impact, str) and financial_impact.strip()==""):
        financial_impact_text = "Financial impact discussed qualitatively; no clear quantified loss or gain disclosed."
    elif isinstance(financial_impact, dict):
        # convert small dict to short sentence
        financial_impact_text = json.dumps(financial_impact, ensure_ascii=False)
    else:
        financial_impact_text = str(financial_impact)

    key_metrics = out.get("key_metrics", None)
    if not key_metrics:
        key_metrics_text = "Key financial metrics are mentioned but not sufficiently detailed to compute precise ratios."
    else:
        # if dict -> convert to short key:val; otherwise string
        if isinstance(key_metrics, dict):
            km_parts = []
            for k,v in key_metrics.items():
                km_parts.append(f"{k}: {v}")
            key_metrics_text = "; ".join(km_parts)
        else:
            key_metrics_text = str(key_metrics)

    critical_dates = out.get("critical_dates", None)
    if not critical_dates:
        critical_dates_text = "No specific critical dates disclosed."
    else:
        if isinstance(critical_dates, list):
            critical_dates_text = ", ".join(map(str, critical_dates))
        else:
            critical_dates_text = str(critical_dates)

    analysis = (out.get("analysis") or "").strip()
    if not analysis:
        analysis = "No explicit analysis provided. Inferred risks include liquidity and leverage concerns; review figures in the text."

    cats_text = "; ".join(risk_categories) if risk_categories else "NONE"

    report = (
        f"risk_severity: {risk_severity}\n"
        f"risk_categories: {cats_text}\n"
        f"financial_impact: {financial_impact_text}\n"
        f"key_metrics: {key_metrics_text}\n"
        f"critical_dates: {critical_dates_text}\n"
        f"analysis: {analysis}"
    )
    return report

# ---------- 5) convert train_split, val_split, and original test into required format ----------
def convert_dataset_to_multiline(ds_split):
    new_records = []
    for ex in tqdm(ds_split):
        instruction = INSTRUCTION
        input_text = ex.get("input","") or ""
        output_obj = ex.get("output", {})  # usually dict
        out_text = dict_to_risk_text(output_obj)
        new_records.append({
            "instruction": instruction,
            "input": input_text,
            "output": out_text
        })
    return new_records

train_final = convert_dataset_to_multiline(train_split)
val_final   = convert_dataset_to_multiline(val_split)
test_final  = convert_dataset_to_multiline(test)

print("Converted sizes -> train:", len(train_final), "val:", len(val_final), "test:", len(test_final))

# ---------- 6) merge train_final + test_final -> train_all_951 ----------
train_all_951 = train_final + test_final
print("Merged train_all_951 size:", len(train_all_951))

# ---------- 7) save as jsonl (one JSON object per line; output contains newlines inside the string) ----------
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            # ensure JSONL line (string fields may contain newlines; json.dumps handles that)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(train_final, f"{OUT_DIR}/train.jsonl")
save_jsonl(val_final,   f"{OUT_DIR}/val.jsonl")
save_jsonl(test_final,  f"{OUT_DIR}/test.jsonl")
save_jsonl(train_all_951, f"{OUT_DIR}/train_all_951.jsonl")

print("Saved files in", OUT_DIR)
